# 03. Short summary vs chunk-specific context

Продолжение эксперимента 02. Чанкинг и `chunk_id` фиксированы, gold-чанки те же.

Гипотеза: короткое саммари и контекст конкретного чанка уменьшат путаницу
разделов и при этом не будут выходить за 512 токенов e5.

Варианты:

1. `baseline` — исходный чанк;
2. `header` — `Article — Section`;
3. `summary_short` — короткое русское саммари раздела (≤ 20 слов);
4. `chunk_context` — контекст конкретного чанка (≤ 35 слов);
5. `combined` — короткое саммари + контекст чанка.

Бюджет: контекст обрезается так, чтобы `context + original ≤ 512` токенов.
Исходный чанк не меняется.


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT)


repo: D:\diffeq-wiki-rag


In [2]:
import json

import pandas as pd
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from qdrant_client.models import Distance, VectorParams
from transformers import AutoTokenizer

from app.ingest.chunking import load_corpus
from app.rag.store import create_qdrant_client, get_embeddings
from eval.contexts import (
    collect_chunks,
    fit_context,
    generate_chunk_contexts,
    generate_short_section_summaries,
)
from eval.metrics import (
    evaluate_retrieval,
    load_gold_chunk_map,
    load_golden,
    summarize,
    summarize_by_topic,
)
from eval.summaries import collect_sections

items = load_golden()
corpus = load_corpus("data/corpus.jsonl")
gold_map = load_gold_chunk_map()
sections = collect_sections(corpus)
chunks = collect_chunks(corpus)
tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")
print("вопросов:", len(items), "| чанков:", len(corpus), "| разделов:", len(sections))


D:\diffeq-wiki-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


вопросов: 30 | чанков: 132 | разделов: 44


## 1. Короткие саммари разделов

Одно короткое описание на раздел (≤ 20 слов), кэшируется в
`eval/results/section_summaries_short.json`.


In [3]:
short_summaries = generate_short_section_summaries(sections)
print("коротких саммари:", len(short_summaries))


short summary Lipschitz continuity::Overview: Раздел даёт общее определение липшицевой непрерывности и её место среди других с...


short summary Lipschitz continuity::Definitions: Раздел формально определяет липшицеву непрерывность через метрики и константу K,...


short summary Lipschitz continuity::Examples: Раздел приводит примеры функций, иллюстрирующие различные случаи соблюдения или ...


short summary Lipschitz continuity::Properties: Раздел описывает аналитические свойства липшицевых функций (дифференцируемость, ...


short summary Lipschitz continuity::Lipschitz manifolds: Раздел определяет липшицевы многообразия через билипшицевы карты и локально липш...


short summary Lipschitz continuity::One-sided Lipschitz: Раздел определяет одностороннюю липшицевость через скалярное произведение и пока...


... (34 outputs left out)

short summary Wronskian::The Wronskian and linear independence: Раздел объясняет связь вронскиана с линейной зависимостью функций и условия, при...


short summary Wronskian::Application to linear differential equations: Раздел объясняет использование вронскиана для нахождения недостающих решений лин...


short summary Wronskian::Generalized Wronskians: Обобщённые вронскианы обобщают определитель Вронского на функции нескольких пере...


short summary Wronskian::History: Раздел History описывает происхождение вронскиана и его название, отличаясь от м...
коротких саммари: 44


In [4]:
for key in [
    "Picard–Lindelöf theorem::Theorem",
    "Wronskian::Definition",
    "Integrating factor::Use",
]:
    print("==", key)
    print(short_summaries.get(key, ""))


== Picard–Lindelöf theorem::Theorem
Формулирует условия существования и единственности решения задачи Коши, в отличие от других разделов, посвящённых доказательству или примерам.
== Wronskian::Definition
Раздел даёт формальное определение вронскиана через определитель матрицы из функций и их производных.
== Integrating factor::Use
Раздел описывает практическое применение интегрирующего множителя для решения дифференциальных уравнений, в отличие от других разделов, посвящённых теории.


## 2. Chunk-specific context

Для каждого из 132 чанков — 1–2 предложения, которые объясняют, что именно
содержит чанк и как он соотносится с разделом. Кэш:
`eval/results/chunk_contexts.json`.


In [5]:
chunk_contexts = generate_chunk_contexts(corpus, short_summaries)
print("контекстов чанков:", len(chunk_contexts))


context Lipschitz continuity::Overview::0: Этот чанк из раздела Overview вводит интуитивное определение липшицевой непрерыв...


context Lipschitz continuity::Overview::1: Этот чанк из раздела Overview связывает липшицеву непрерывность с теоремой Пикар...


context Lipschitz continuity::Definitions::0: Этот чанк открывает раздел Definitions и вводит формальное определение липшицево...


context Lipschitz continuity::Definitions::1: Этот чанк из раздела Definitions приводит формулу |f(x₁)−f(x₂)| ≤ K|x₁−x₂| для в...


context Lipschitz continuity::Definitions::2: Этот чанк из раздела Definitions вводит локальную липшицеву непрерывность через ...


context Lipschitz continuity::Definitions::3: Этот чанк в разделе Definitions вводит понятие K-билипшицева отображения через д...


... (122 outputs left out)

context Wronskian::Application to linear differential equations::0: Этот чанк из раздела о применении вронскиана к линейным дифференциальным уравнен...


context Wronskian::Application to linear differential equations::1: Этот чанк находится в начале раздела и обобщает метод на уравнения высших порядк...


context Wronskian::Generalized Wronskians::0: Этот чанк в разделе Generalized Wronskians вводит определение обобщённого вронск...


context Wronskian::History::0: Этот чанк из раздела History содержит сведения о происхождении вронскиана: его в...
контекстов чанков: 132


In [6]:
for chunk_id in [
    "Picard–Lindelöf theorem::Theorem::0",
    "Wronskian::Definition::0",
    "Integrating factor::Use::Solving first order linear ordinary differential equations::0",
]:
    print("==", chunk_id)
    print(chunk_contexts.get(chunk_id, ""))


== Picard–Lindelöf theorem::Theorem::0
Этот чанк содержит формальную формулировку теоремы Пикара–Линделёфа: условия на прямоугольник D, точку (t₀,y₀) и функцию f (непрерывность по t, липшицевость по y), а также вывод о существовании и единственности решения задачи Коши на интервале [t₀−ε, t₀+ε]. Он открывает раздел Theorem, задавая условия теоремы до её доказательства.
== Wronskian::Definition::0
Этот чанк из раздела Definition вводит формальное определение вронскиана: сначала для двух функций как W(f,g)=fg'−gf', затем в общем виде как определитель матрицы из n функций и их производных до (n−1)-го порядка на интервале I. В конце чанк упоминает, что для решений линейного дифференциального уравнения вронскиан можно найти через тождество Абеля.
== Integrating factor::Use::Solving first order linear ordinary differential equations::0
Этот чанк из раздела «Use» вводит каноническую форму линейного дифференциального уравнения первого порядка y'+P(x)y=Q(x) и определяет интегрирующий множитель 

## 3. Варианты и бюджет токенов

Контекст обрезается, если `original + context > 512`. Исходный текст чанка
остаётся нетронутым.


In [7]:
def make_docs(kind: str) -> list[Document]:
    documents = []
    for source in corpus:
        meta = source.metadata
        original = source.page_content
        chunk_id = meta["chunk_id"]
        section_key = f"{meta['article']}::{meta['section']}"

        if kind == "baseline":
            raw_context = ""
        elif kind == "header":
            raw_context = f"{meta['article']} — {meta['section']}"
        elif kind == "summary_short":
            raw_context = short_summaries.get(section_key, "")
        elif kind == "chunk_context":
            raw_context = chunk_contexts.get(chunk_id, "")
        elif kind == "combined":
            raw_context = (
                short_summaries.get(section_key, "")
                + "\n"
                + chunk_contexts.get(chunk_id, "")
            )
        else:
            raise ValueError(kind)

        context, truncated = fit_context(original, raw_context, tokenizer)
        embedding_text = f"{context}\n\n{original}" if context else original

        documents.append(
            Document(
                page_content=embedding_text,
                metadata={
                    **meta,
                    "original_content": original,
                    "variant": kind,
                    "context_truncated": truncated,
                    "context_tokens": len(
                        tokenizer.encode(context, add_special_tokens=False)
                    ),
                },
            )
        )
    return documents


variants = {
    kind: make_docs(kind)
    for kind in ["baseline", "header", "summary_short", "chunk_context", "combined"]
}
for kind, documents in variants.items():
    truncated = sum(1 for document in documents if document.metadata["context_truncated"])
    print(kind, "| chunks:", len(documents), "| context truncated:", truncated)


baseline | chunks: 132 | context truncated: 0
header | chunks: 132 | context truncated: 1
summary_short | chunks: 132 | context truncated: 4
chunk_context | chunks: 132 | context truncated: 7
combined | chunks: 132 | context truncated: 9


## 4. Временные коллекции Qdrant

In [8]:
client = create_qdrant_client()
embedding = get_embeddings()
stores = {}

for kind, documents in variants.items():
    collection = f"ctx_{kind}"
    if client.collection_exists(collection):
        client.delete_collection(collection)
    client.create_collection(
        collection_name=collection,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )
    store = QdrantVectorStore(
        client=client,
        collection_name=collection,
        embedding=embedding,
    )
    store.add_documents(documents, batch_size=64)
    stores[kind] = store
    print("indexed:", kind, len(documents))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  48%|████▊     | 96/199 [00:00<00:00, 845.76it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1366.25it/s]

indexed: baseline 132


indexed: header 132


indexed: summary_short 132


indexed: chunk_context 132


indexed: combined 132


## 5. Метрики

In [9]:
frames = {}
rows = []
for kind, store in stores.items():
    frame = evaluate_retrieval(store, items, corpus, k=5, gold_map=gold_map)
    frames[kind] = frame
    rows.append({"variant": kind, **summarize(frame)})

metrics_df = pd.DataFrame(rows)
metrics_df


,variant,questions,article_hit@1,section_hit@1,recall@1,precision@1,article_hit@3,section_hit@3,recall@3,precision@3,article_hit@5,section_hit@5,recall@5,precision@5,mrr
0,baseline,30,0.7333,0.3000,0.1056,0.1667,0.8333,0.5000,0.2444,0.1444,0.9000,0.7000,0.3722,0.1467,0.7872
1,header,30,0.6667,0.3667,0.1222,0.2333,0.8000,0.6333,0.3500,0.2222,0.9333,0.7667,0.4556,0.1667,0.7650
2,summary_short,30,0.8667,0.4667,0.1167,0.2000,0.9667,0.6000,0.3111,0.2111,1.0000,0.8000,0.5111,0.2067,0.9194
3,chunk_context,30,0.8000,0.4333,0.1722,0.3000,0.9000,0.6333,0.4111,0.2667,0.9333,0.8000,0.5611,0.2200,0.8528
4,combined,30,0.8333,0.4667,0.1389,0.2333,0.9333,0.8000,0.5167,0.3333,1.0000,0.9000,0.6556,0.2600,0.8944


## 6. Токены и обрезка контекста

In [10]:
token_rows = []
for kind, documents in variants.items():
    lengths = [
        len(tokenizer.encode(document.page_content, add_special_tokens=False))
        for document in documents
    ]
    token_rows.append(
        {
            "variant": kind,
            "min": min(lengths),
            "p50": int(pd.Series(lengths).median()),
            "mean": round(sum(lengths) / len(lengths), 1),
            "max": max(lengths),
            "over_512": sum(1 for value in lengths if value > 512),
            "context_truncated": sum(
                1 for document in documents if document.metadata["context_truncated"]
            ),
        }
    )
pd.DataFrame(token_rows)


,variant,min,p50,mean,max,over_512,context_truncated
0,baseline,10,213,216.7,507,0,0
1,header,16,226,228.7,510,0,1
2,summary_short,37,253,256.5,510,0,4
3,chunk_context,70,297,292.6,510,0,7
4,combined,97,337,331.0,510,0,9


## 7. Recall@5 по темам

In [11]:
topic_recall = {}
for kind, frame in frames.items():
    topic_recall[kind] = summarize_by_topic(frame).set_index("topic")["recall@5"]
pd.DataFrame(topic_recall).round(3)


,baseline,header,summary_short,chunk_context,combined
topic,,,,,
integrating_factor,0.000,0.167,0.500,0.500,0.667
linear_ode,0.633,0.533,0.233,0.733,0.700
lipschitz,0.208,0.292,0.292,0.375,0.417
multi_hop,0.433,0.467,0.500,0.633,0.700
ode,0.250,0.250,0.750,0.500,0.875
picard,0.000,0.500,0.400,0.200,0.300
wronskian,1.000,0.875,1.000,1.000,1.000


## 8. Сохранение

In [12]:
payload = {
    "variants": metrics_df.to_dict(orient="records"),
    "per_topic_recall5": {
        kind: {topic: float(value) for topic, value in values.items()}
        for kind, values in topic_recall.items()
    },
    "token_stats": token_rows,
    "caches": {
        "short_summaries": "eval/results/section_summaries_short.json",
        "chunk_contexts": "eval/results/chunk_contexts.json",
    },
}
output = Path("eval/results/chunk_context_experiment.json")
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("сохранено:", output)


сохранено: eval\results\chunk_context_experiment.json


## 9. Как читать

- `over_512` должен быть 0 — мы контролируем бюджет;
- `context_truncated` показывает, у скольких чанков контекст пришлось обрезать;
- сравниваем `summary_short`, `chunk_context` и `combined` с `baseline` и `header`;
- главная метрика — `recall@5`, затем `section_hit@3`.
